# Transformers and Attention Mechanisms
The Transformer (Vaswani et al., 2017) revolutionized deep learning by replacing recurrence with **attention**, enabling full parallelization. This notebook covers self-attention, multi-head attention, the full Transformer architecture, BERT, GPT, T5, and Vision Transformer (ViT).

## 1. The Attention Mechanism
Given queries (Q), keys (K), and values (V):

```
Attention(Q, K, V) = softmax( (Q · K^T) / sqrt(d_k) ) · V
```
- Q·K^T measures similarity between each query and every key
- Dividing by sqrt(d_k) prevents dot products from exploding
- Softmax converts scores to attention weights that sum to 1
- Output is a weighted sum of values V

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = tf.cast(tf.shape(k)[-1], tf.float32)
    scores = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(d_k)
    if mask is not None:
        scores += (mask * -1e9)
    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, v), weights

Q = tf.random.normal((1, 2, 4))
K = tf.random.normal((1, 3, 4))
V = tf.random.normal((1, 3, 4))
out, w = scaled_dot_product_attention(Q, K, V)
print("Attention output shape:", out.shape)
print("Attention weights (rounded):", w.numpy().round(3))

Attention output shape: (1, 2, 4)
Attention weights (rounded): [[[0.628 0.359 0.013]
  [0.314 0.024 0.663]]]


## 2. Multi-Head Attention
Projects Q, K, V into h heads, runs attention independently in each, then concatenates and projects:

```
MultiHead(Q,K,V) = Concat(head_1, ..., head_h) · W^O
head_i = Attention(Q*W_i^Q,  K*W_i^K,  V*W_i^V)
```
Different heads learn to attend to different relationships simultaneously (syntax, semantics, coreference etc.).

In [2]:
mha = layers.MultiHeadAttention(num_heads=8, key_dim=64)
x = tf.random.normal((2, 20, 512))   # (batch, seq_len, d_model)
out = mha(x, x, x)                   # Self-attention: Q=K=V=x
print("Multi-Head Attention output:", out.shape)  # (2, 20, 512)

Multi-Head Attention output: (2, 20, 512)


## 3. The Transformer Architecture
**Encoder Layer (BERT-style):**
1. Multi-Head Self-Attention
2. Add & Layer Norm
3. Feed-Forward Network (FFN: Linear → ReLU → Linear)
4. Add & Layer Norm

**Decoder Layer (GPT-style):**
1. Masked Causal Self-Attention
2. Add & Layer Norm
3. Cross-Attention (Q=decoder, K/V=encoder output)
4. Add & Layer Norm
5. FFN + Add & Layer Norm

Sinusoidal Positional Encodings inject sequence order since attention is naturally order-agnostic.

## 4. BERT
- Encoder-only Transformer
- Bidirectional: sees full left+right context
- Pre-trained via Masked Language Modeling (MLM) and Next Sentence Prediction (NSP)
- Fine-tuned for classification, NER, QA

## 5. GPT
- Decoder-only Transformer with causal (left-to-right) masking
- Pre-trained via next-token prediction (Causal Language Modeling)
- Excellent for text generation; GPT-3/4 emerge impressive zero-shot abilities at scale

## 6. T5
- Full Encoder-Decoder Transformer
- Every NLP task reformulated as text-to-text: "translate English to German: ..."
- Pre-trained with span corruption objective

## 7. Vision Transformer (ViT)
- Images split into fixed 16x16 pixel patches
- Each patch linearly projected to a d_model-dimensional embedding ("visual tokens")
- Standard Transformer Encoder applied; [CLS] token embedding fed to classifier
- Outperforms CNNs at scale (100M+ images)

In [3]:
# ViT-style patch embedding
def vit_patch_embedding(image_size=224, patch_size=16, d_model=768):
    num_patches = (image_size // patch_size) ** 2
    inputs = layers.Input(shape=(image_size, image_size, 3))
    x = layers.Conv2D(d_model, kernel_size=patch_size,
                      strides=patch_size, padding='valid')(inputs)
    x = layers.Reshape((num_patches, d_model))(x)
    return models.Model(inputs, x, name="PatchEmbedding")

pe = vit_patch_embedding()
pe.summary()

Model: "PatchEmbedding"

┏━━━━━━━━━┳━━━━━━━┳━━━━┓
┃ Layer   ┃ Outp… ┃ P… ┃
┃ (type)  ┃ Shape ┃  # ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━┩
│ input_… │ (Non… │  0 │
│ (Input… │ 224,  │    │
│         │ 224,  │    │
│         │ 3)    │    │
├─────────┼───────┼────┤
│ conv2d  │ (Non… │ 5… │
│ (Conv2… │ 14,   │    │
│         │ 14,   │    │
│         │ 768)  │    │
├─────────┼───────┼────┤
│ reshape │ (Non… │  0 │
│ (Resha… │ 196,  │    │
│         │ 768)  │    │
└─────────┴───────┴────┘

 Total params: 590,592 (2.25 MB)

 Trainable params: 590,592 (2.25 MB)

 Non-trainable params: 0 (0.00 B)

# Conclusions and Key Takeaways
- The Transformer showed that attention alone suffices — recurrence is unnecessary.
- Scale consistently yields better models: more layers, larger d_model, more heads.
- Pre-train + Fine-tune is the dominant NLP/CV paradigm.
- The architecture is domain-agnostic: ViT applies identical code to vision with patches replacing words.

# Pros and Cons
**Pros:**
- Fully parallelizable training — massive GPU utilization vs sequential RNNs
- Global receptive field: any two tokens interact in O(1) layers
- Highly transferable via large-scale unsupervised pre-training
- Flexible: encoder-only (BERT), decoder-only (GPT), or encoder-decoder (T5)

**Cons:**
- O(n^2) memory and compute in sequence length — problematic for very long documents
- Requires massive pre-training data to be effective from scratch
- Inference is more expensive than RNNs for streaming/real-time applications
- Positional encodings are non-trivial; extrapolation to longer sequences is difficult

# 15 Interview Questions and Answers

1. **What is the core formula for Scaled Dot-Product Attention?**
   *Answer*: Attention(Q,K,V) = softmax(Q·K^T / sqrt(d_k)) · V. The scaling by sqrt(d_k) prevents very large dot products that would push the softmax into tiny-gradient regions.

2. **Why do we scale by sqrt(d_k)?**
   *Answer*: As d_k grows, the variance of the dot products grows, pushing softmax into regions with vanishing gradients. Scaling stabilizes training.

3. **What is Multi-Head Attention and why is it used?**
   *Answer*: Multiple attention heads each project Q/K/V to a lower-dimensional space and attend independently. Their outputs are concatenated and projected back, allowing the model to jointly attend to information from different representation subspaces simultaneously.

4. **What is Positional Encoding?**
   *Answer*: Added to input embeddings to inject token order information, since self-attention is permutation-invariant. Standard Transformers use sinusoidal functions; recent variants learn the encodings.

5. **What is Masked Self-Attention in GPT?**
   *Answer*: A causal mask zeros out attention to future positions, ensuring autoregressive generation: when predicting token t, only tokens 1 through t-1 are visible.

6. **What is Cross-Attention in the Transformer Decoder?**
   *Answer*: An attention layer where Keys and Values come from the encoder output and Queries come from the decoder's own hidden state, allowing the decoder to selectively focus on relevant parts of the encoded source.

7. **How does BERT differ from GPT?**
   *Answer*: BERT uses an encoder-only architecture seeing full bidirectional context, pre-trained with masked token prediction. GPT uses decoder-only with causal left-to-right masking, pre-trained with next-token prediction.

8. **Why can't BERT be used for open-ended text generation?**
   *Answer*: BERT is bidirectional — each token's representation already incorporates future tokens. Autoregressive generation requires predicting the next token using only past context, which BERT's architecture cannot do.

9. **What is a Vision Transformer (ViT)?**
   *Answer*: An image is divided into 16×16 patches, each linearly projected to an embedding. These patch embeddings + positional encodings are fed into a standard Transformer encoder for image classification.

10. **What is the Feed-Forward sublayer in a Transformer block?**
    *Answer*: A two-layer MLP applied position-wise: FFN(x) = max(0, xW_1+b_1)W_2+b_2. The inner dimension is typically 4x d_model. It adds non-linear transformation capacity after the attention mechanism.

11. **Why does Transformer use Layer Norm instead of Batch Norm?**
    *Answer*: Layer Norm normalizes across the feature dimension for each sample independently. Batch Norm normalizes across the batch, which breaks when batch sizes are small or sequence lengths vary.

12. **What is the time/memory complexity of self-attention?**
    *Answer*: O(n^2 * d) where n is sequence length and d is model dim. The quadratic n^2 term is the key bottleneck for very long sequences (>4096 tokens).

13. **How does T5 differ from BERT and GPT?**
    *Answer*: T5 uses the full encoder-decoder architecture and unifies every NLP task under a text-to-text framework where both inputs and outputs are plain text strings.

14. **What does the [CLS] token represent in BERT?**
    *Answer*: A special token prepended to every input whose final hidden state aggregates the full sequence representation and is used as input to task-specific classification heads.

15. **What is Flash Attention?**
    *Answer*: An IO-aware algorithm that tiles the attention computation to keep data in fast SRAM on the GPU, reducing memory reads/writes to HBM. This makes attention 2-4x faster and reduces memory from O(n^2) to O(n), enabling much longer contexts.
